In [1]:
import sys, re, json
sys.path.append('..') 

from scripts.constants import *
from scripts.utils import *
from scripts.sedona_config import *

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [2]:
project_crs = 'EPSG:27700'

T3_30_300_DIR = VECTOR_OUT_DIR / "3-30-300"
T3_dir = T3_30_300_DIR / "T3"
T30_dir = T3_30_300_DIR / "T30"
T300_dir = T3_30_300_DIR / "T300"
trees_dir = T3_30_300_DIR / "VOM_Trees"
vom_dir = RASTER_IN_DIR / "Defra" / "VOM"
vom_lad_dir = vom_dir / "LADs"
t3_30_300_path = T3_30_300_DIR / "T3_30_300.geojson"
imd_lsoa_bua_boundaries_path = VECTOR_OUT_DIR / "IMD" / "English_IMD_2019_BUA_filtered_boundaries.geojson"
imd_england_path = VECTOR_IN_DIR / "IMD" / "English IMD 2019" / "IMD_2019.shp"
os_5km_boundaries_path = VECTOR_IN_DIR / "OS" / "National_Grid" / "5km_grid_region.shp"
buildings_path = VECTOR_IN_DIR / "EDINA" / "Buildings_6183" / "Buildings_6183.parquet"
chm_lad_tiles_path = vom_lad_dir / "LAD_CHM_tiles_paths.json"

In [3]:
chm_lad_tiles_dict = json.load(open(chm_lad_tiles_path))
os_5km_boundaries_gdf = gpd.read_file(os_5km_boundaries_path).to_crs(project_crs)
imd_lsoa_bua_gdf = gpd.read_file(imd_lsoa_bua_boundaries_path)

In [4]:
geo_level = 'LAD22CD'
london_codes = imd_lsoa_bua_gdf[imd_lsoa_bua_gdf['RGN22CD'] == 'E12000007'][geo_level].unique().tolist()
cambridge_code = 'E07000008'

In [5]:
def extract_grid_reference(filename: str) -> str|None:
    """
    Extracts a grid reference from a given filename.
    The function searches for a pattern in the filename that matches 'VOM' or 'VOM_HS'
    followed by an underscore, a two-letter code, a four-digit number, and another underscore.
    If such a pattern is found, it returns the grid reference (the two-letter code and the four-digit number).
    If no match is found, it returns None.
    Parameters:
        filename (str | Path): The name of the file from which to extract the grid reference.
    Returns:
        str | None: The extracted grid reference if a match is found, otherwise None.
    """

    match = re.search(r'VOM_([A-Z]{2}\d{4})_', filename)
    if match:
        return match.group(1)
    return None

def check_tree_vom_pair(chm_path: str|Path, trees_dir: str|Path) -> bool:
    """
    Check if a CHM file has a corresponding tree file.
    Parameters:
        chm_path (str | Path): The path to the CHM file.
        trees_dir (str | Path): The directory containing the tree files.
    Returns:
        bool: True if a corresponding tree file exists, otherwise False.
    """

    tile_name = extract_grid_reference(chm_path)
    chm_path = chm_path if isinstance(chm_path, Path) else Path(chm_path)
    year = chm_path.parent.name

    trees_path = trees_dir / f"VOM_trees_{tile_name}_{year}.gpkg"

    if trees_path.exists():
        return str(trees_path)
    
def get_overlapping_tiles(imd_lsoa_bua_buffer_gdf, os_5km_boundaries_gdf, geo_level, geo_code):
    # Select one feature from imd_lsoa_bua_buffer_gdf
    selected_feature = imd_lsoa_bua_buffer_gdf[imd_lsoa_bua_buffer_gdf[geo_level] == geo_code]

    # Get the overlapping features
    overlapping_tiles_lst = gpd.overlay(selected_feature, os_5km_boundaries_gdf, how='intersection')['TILE_NAME'].unique().tolist()

    return overlapping_tiles_lst

def get_vom_trees_paths(overlapping_tiles_lst: list, vom_tree_pair_dict: dict) -> list:

    vom_trees_path_lst = [path_pair[1] for _,v in vom_tree_pair_dict.items() for path_pair in v if path_pair[1] is not None]
    for tile_name in overlapping_tiles_lst:
        translated_tile_name = translate_tile_name(tile_name).upper()
        trees_path_lst = [path for path in vom_trees_path_lst if translated_tile_name in path]

        return trees_path_lst

In [95]:
import shutil
from pathlib import Path

def copy_files(file_list, destination_folder):
    """
    Copy a list of files to a destination folder.
    
    Parameters:
        file_list (list): List of file paths to be copied.
        destination_folder (str | Path): The destination folder where files will be copied.
    """
    destination_folder.mkdir(parents=True, exist_ok=True)
    
    for file_path in file_list:
        try:
            file_path = Path(file_path)
            if file_path.exists():
                shutil.copy(file_path, destination_folder / file_path.name)
            else:
                print(f"File {file_path} does not exist.")
        except Exception as e:
            print(e)

In [28]:
vom_tree_pair_dict = {k: [(chm_path, check_tree_vom_pair(chm_path, trees_dir)) for chm_path in v] for k, v in chm_lad_tiles_dict.items()}

In [93]:
tree_pairs = set([item for sublist in [v for k, v in vom_tree_pair_dict.items() if k in london_codes] for item in sublist])
files = list(set([i[1] for i in tree_pairs if i[1] is not None]))
files.sort()
files

['/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP5515_2023.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP5535_2019.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP6005_2023.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP6010_2023.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP6015_2023.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP6020_2023.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP6025_2019.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP6030_2019.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP6035_2019.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP6040_2019.gpkg',
 '/maps/acz25/phd-thesis-data/output/vector/3-30-300/VOM_Trees/VOM_trees_SP6505_

In [94]:
city = 'London'
destination_folder = DATA_DIR.parent / "example-data" / city / "VOM_trees"
copy_files(files, destination_folder)

In [96]:
destination_folder = DATA_DIR.parent / "example-data" / "London" / "VOM_trees"
len(list(destination_folder.glob('*.gpkg')))

265

In [6]:
os.environ["JAVA_HOME"] = JAVA_HOME
sedona = get_spark()

24/12/17 15:35:55 WARN Utils: Your hostname, kinabalu resolves to a loopback address: 127.0.1.1; using 128.232.93.1 instead (on interface eno12399np0)
24/12/17 15:35:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
https://artifacts.unidata.ucar.edu/repository/unidata-all added as a remote repository with the name: repo-1
Ivy Default Cache set to: /home/acz25/.ivy2/cache
The jars for the packages stored in: /home/acz25/.ivy2/jars
org.apache.sedona#sedona-spark-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
net.postgis#postgis-jdbc added as a dependency
net.postgis#postgis-geometry added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2e677d43-0dbc-4662-a187-3fc5a27e9b4b;1.0
	confs: [default]


:: loading settings :: url = jar:file:/maps-priv/maps/acz25/miniconda3/envs/3-30-300-env/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.sedona#sedona-spark-3.5_2.12;1.7.0 in central
	found org.apache.sedona#sedona-common;1.7.0 in central
	found org.apache.commons#commons-math3;3.6.1 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.wololo#jts2geojson;0.16.1 in central
	found org.locationtech.spatial4j#spatial4j;0.8 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-annotations;1.1 in central
	found org.codehaus.mojo#animal-sniffer-annotations;1.14 in central
	found com.uber#h3;4.1.1 in central
	found net.sf.geographiclib#GeographicLib-Java;1.52 in central
	found com.github.ben-manes.caffeine#caffeine;2.9.2 in central
	found org.checkerframework#checker-qual;3.10.0 in central
	found com.google.error

In [7]:
boundaries_sdf = sedona.createDataFrame(imd_lsoa_bua_gdf.drop(columns=['LSOA21NMW', 'LAD22NMW', 'BUA22NMG', 'BUA22NMW', 'RGN22NMW'], axis=1))
boundaries_sdf.createOrReplaceTempView('boundaries')
buildings_sdf = sedona.read.format("geoparquet").load(str(buildings_path))
buildings_sdf.createOrReplaceTempView("buildings")

In [8]:
# geo_level = 'RGN22CD'
# geo_code = 'E12000007'
# city = 'London'
geo_level = 'LAD22CD'
geo_code = cambridge_code
city = 'Cambridge'

In [9]:
geo_boundary_sdf = sedona.sql(
    f"""
        SELECT ST_Union_Aggr(geometry) AS geometry
        FROM boundaries
        WHERE {geo_level} = '{geo_code}'
    """)
geo_boundary_sdf.createOrReplaceTempView("geo_boundary")

geo_buildings_sdf = sedona.sql(
    """
        SELECT b.* FROM buildings b, geo_boundary g 
        WHERE ST_Intersects(b.geometry, g.geometry)
    """)
geo_buildings_sdf.createOrReplaceTempView("geo_buildings")

In [12]:
output_path = DATA_DIR.parent / "example-data" / city / "Buildings" / "buildings.geojson"
output_path.parent.mkdir(parents=True, exist_ok=True)
geo_buildings_sdf.write.format("geojson").save(str(output_path))#.format("geoparquet").save(str(output_path))